In [1]:
import requests
import pandas as pd
from datetime import datetime

# URL de la API
url = 'https://api.exchangerate-api.com/v4/latest/USD'

# Realizar la solicitud GET
response = requests.get(url)

# Verificar que la solicitud fue exitosa
if response.status_code == 200:
    # Convertir la respuesta JSON a un diccionario
    data = response.json()
    
    # Convertir los datos de tasas de cambio en un DataFrame
    rates_df = pd.DataFrame(list(data['rates'].items()), columns=['Currency', 'Rate'])
    
    # Filtrar las monedas específicas
    specific_currencies = {'USD', 'ARS', 'AUD', 'BDT', 'BRL', 'CAD', 'CNY', 'ETB', 'EUR', 'FJD', 'GBP', 'HTG', 'JPY', 'NGN', 'NIO', 'PGK', 'PYG', 'SLL', 'VUV', 'ZAR'}
    rates_df = rates_df[rates_df['Currency'].isin(specific_currencies)]

    # Agregar columna temporal para el control de ingesta de datos
    rates_df['Ingestion_Time'] = datetime.now()

    # Diccionario de monedas a datos geográficos
    currency_to_geo = {
        'USD': {'Country': 'United States', 'Region': 'North America', 'Continent': 'America'},
        'CAD': {'Country': 'Canada', 'Region': 'North America', 'Continent': 'America'},
        'HTG': {'Country': 'Haiti', 'Region': 'Caribbean', 'Continent': 'America'},
        'NIO': {'Country': 'Nicaragua', 'Region': 'Central America', 'Continent': 'America'},
        'BRL': {'Country': 'Brazil', 'Region': 'South America', 'Continent': 'America'},
        'ARS': {'Country': 'Argentina', 'Region': 'South America', 'Continent': 'America'},
        'AUD': {'Country': 'Australia', 'Region': 'Oceania', 'Continent': 'Oceania'},
        'BDT': {'Country': 'Bangladesh', 'Region': 'South Asia', 'Continent': 'Asia'},
        'EUR': {'Country': 'Eurozone', 'Region': 'Europe', 'Continent': 'Europe'},
        'GBP': {'Country': 'United Kingdom', 'Region': 'Europe', 'Continent': 'Europe'},
        'FJD': {'Country': 'Fiji', 'Region': 'Melanesia', 'Continent': 'Oceania'},
        'JPY': {'Country': 'Japan', 'Region': 'East Asia', 'Continent': 'Asia'},
        'CNY': {'Country': 'China', 'Region': 'East Asia', 'Continent': 'Asia'},
        'NGN': {'Country': 'Nigeria', 'Region': 'West Africa', 'Continent': 'Africa'},
        'ETB': {'Country': 'Ethiopia', 'Region': 'East Africa', 'Continent': 'Africa'},
        'PGK': {'Country': 'Papua New Guinea', 'Region': 'Melanesia', 'Continent': 'Oceania'},
        'PYG': {'Country': 'Paraguay', 'Region': 'South America', 'Continent': 'America'},
        'SLL': {'Country': 'Sierra Leone', 'Region': 'West Africa', 'Continent': 'Africa'},
        'VUV': {'Country': 'Vanuatu', 'Region': 'Melanesia', 'Continent': 'Oceania'},
        'ZAR': {'Country': 'South Africa', 'Region': 'Southern Africa', 'Continent': 'Africa'}
    }

    # Agregar datos geográficos usando el diccionario
    geo_data = rates_df['Currency'].map(currency_to_geo)
    rates_df['Country'] = geo_data.apply(lambda x: x['Country'] if isinstance(x, dict) and 'Country' in x else None)
    rates_df['Region'] = geo_data.apply(lambda x: x['Region'] if isinstance(x, dict) and 'Region' in x else None)
    rates_df['Continent'] = geo_data.apply(lambda x: x['Continent'] if isinstance(x, dict) and 'Continent' in x else None)
    
    # Definir las monedas de los países ricos
    wealthy_currencies = {'USD', 'CAD', 'BRL', 'ARS', 'AUD', 'FJD', 'JPY', 'CNY', 'EUR', 'GBP', 'ZAR', 'ETB'}

    # Agregar columna Wealthy
    rates_df['Wealthy'] = rates_df['Currency'].apply(lambda x: 1 if x in wealthy_currencies else 0)
    
    # Mostrar todas las columnas del DataFrame y las primeras 5 filas
    print("\nPrimeras 5 filas del DataFrame:")
    print(rates_df.head())

    # Hasta aquí es donde se debe detener el código para no enviar los datos a Redshift

else:
    print(f"Failed to retrieve data: {response.status_code}")



Primeras 5 filas del DataFrame:
   Currency    Rate             Ingestion_Time        Country         Region  \
0       USD    1.00 2024-07-30 20:29:00.754365  United States  North America   
7       ARS  932.25 2024-07-30 20:29:00.754365      Argentina  South America   
8       AUD    1.53 2024-07-30 20:29:00.754365      Australia        Oceania   
13      BDT  117.50 2024-07-30 20:29:00.754365     Bangladesh     South Asia   
20      BRL    5.62 2024-07-30 20:29:00.754365         Brazil  South America   

   Continent  Wealthy  
0    America        1  
7    America        1  
8    Oceania        1  
13      Asia        0  
20   America        1  
